# WEEK 3 - Linear Regression part 3: linear regression with forward and backward selection, PCR, and PLSR

In [ ]:
# !pip install statsmodels


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import networkx as nx

In [3]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
#1
df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

#2
df_pima = pd.read_csv("pima_indian_diabetes_dataset.csv") 

# Dataset 1 Foward and Backware Feature Selection

Forward Selection

In [5]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score

# --- 1. Prepare data ---
X = df.drop(columns=['Diabetes_012'])
y = df['Diabetes_012']

# --- 2. Scale and split ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- 3. Logistic Regression model ---
log_reg = LogisticRegression(max_iter=5000, solver='lbfgs', multi_class='ovr')

# --- 4. Forward Feature Selection ---
sfs_forward = SequentialFeatureSelector(
    log_reg,
    n_features_to_select='auto',   # or a fixed number, e.g., 8
    direction='forward',           # 👈 forward selection
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_forward.fit(X_train, y_train)

# --- 5. View selected features ---
selected_forward = X.columns[sfs_forward.get_support()]
print("Forward Selected Features:")
print(selected_forward)


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic

Forward Selected Features:
Index(['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HvyAlcoholConsump',
       'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'PhysHlth'],
      dtype='object')


Backward Feature Selection 

In [6]:
# --- Backward Feature Selection ---

X = df.drop(columns=['Diabetes_012'])
y = df['Diabetes_012']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

log_reg = LogisticRegression(max_iter=5000, solver='lbfgs', multi_class='ovr')

sfs_backward = SequentialFeatureSelector(
    log_reg,
    n_features_to_select='auto',
    direction='backward',          # 👈 backward elimination
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_backward.fit(X_train, y_train)

selected_backward = X.columns[sfs_backward.get_support()]
print("Backward Selected Features:")
print(selected_backward)


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic

Backward Selected Features:
Index(['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack',
       'PhysActivity', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'Education',
       'Income'],
      dtype='object')


| Method                                   | Use Case                                  | Pros                           | Cons                                               |
| ---------------------------------------- | ----------------------------------------- | ------------------------------ | -------------------------------------------------- |
| **Forward**                              | You want a small, efficient model quickly | Fast, simple                   | Might miss a feature that only matters with others |
| **Backward**                             | You start with many variables             | Comprehensive                  | Slower, can be computationally heavy               |
| **Stepwise**                             | You want a mix                            | Balanced approach              | Can overfit if not cross-validated                 |
| **Regularization (Lasso / Elastic Net)** | You have many correlated predictors       | Automatic shrinkage & sparsity | Doesn’t explicitly test feature subsets            |


comparing and evaluating forward and backware at the same time

In [ ]:
# Forward vs Backward Feature Selection – Logistic Regression

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, roc_auc_score

# --- 1. Prepare data ---
X = df.drop(columns=['Diabetes_012'])
y = df['Diabetes_012']

# --- 2. Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- 4. Define base Logistic Regression model ---
log_reg = LogisticRegression(
    solver='lbfgs', max_iter=5000, multi_class='ovr', random_state=42
)

# Forward Feature Selection

sfs_forward = SequentialFeatureSelector(
    log_reg,
    direction='forward',
    n_features_to_select='auto',   # can also set a number like 10
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_forward.fit(X_train, y_train)
forward_features = X.columns[sfs_forward.get_support()]

# Train model on selected features
X_train_fwd = X_train[:, sfs_forward.get_support()]
X_test_fwd = X_test[:, sfs_forward.get_support()]
log_reg.fit(X_train_fwd, y_train)
y_pred_fwd = log_reg.predict(X_test_fwd)
y_prob_fwd = log_reg.predict_proba(X_test_fwd)

forward_acc = accuracy_score(y_test, y_pred_fwd)
forward_auc = roc_auc_score(y_test, y_prob_fwd, multi_class='ovr', average='macro')

# Backward Feature Selection

sfs_backward = SequentialFeatureSelector(
    log_reg,
    direction='backward',
    n_features_to_select='auto',
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_backward.fit(X_train, y_train)
backward_features = X.columns[sfs_backward.get_support()]

# Train model on selected features
X_train_bwd = X_train[:, sfs_backward.get_support()]
X_test_bwd = X_test[:, sfs_backward.get_support()]
log_reg.fit(X_train_bwd, y_train)
y_pred_bwd = log_reg.predict(X_test_bwd)
y_prob_bwd = log_reg.predict_proba(X_test_bwd)

backward_acc = accuracy_score(y_test, y_pred_bwd)
backward_auc = roc_auc_score(y_test, y_prob_bwd, multi_class='ovr', average='macro')

# Results Summary

print("🔹 Forward Selection Features:")
print(list(forward_features))
print(f"Accuracy: {forward_acc:.3f} | AUC: {forward_auc:.3f}\n")

print("🔹 Backward Selection Features:")
print(list(backward_features))
print(f"Accuracy: {backward_acc:.3f} | AUC: {backward_auc:.3f}\n")

# Quick comparison table
results_df = pd.DataFrame({
    'Method': ['Forward', 'Backward'],
    'Selected_Features': [len(forward_features), len(backward_features)],
    'Accuracy': [forward_acc, backward_acc],
    'AUC': [forward_auc, backward_auc]
})

print("🏁 Feature Selection Comparison:")
print(results_df)


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic

🔹 Forward Selection Features:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'PhysHlth']
Accuracy: 0.846 | AUC: 0.763

🔹 Backward Selection Features:
['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'Education', 'Income']
Accuracy: 0.846 | AUC: 0.766

🏁 Feature Selection Comparison:
     Method  Selected_Features  Accuracy       AUC
0   Forward                 10  0.845790  0.763169
1  Backward                 11  0.845829  0.766200


In [10]:
# Forward vs Backward Feature Selection – Logistic Regression - balanced classes

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, roc_auc_score

# --- 1. Prepare data ---
X = df.drop(columns=['Diabetes_012'])
y = df['Diabetes_012']

# --- 2. Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- 4. Define base Logistic Regression model ---
log_reg = LogisticRegression(
    solver='lbfgs', max_iter=5000, multi_class='ovr', random_state=42, class_weight='balanced'
)

# Forward Feature Selection

sfs_forward = SequentialFeatureSelector(
    log_reg,
    direction='forward',
    n_features_to_select='auto',   # can also set a number like 10
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_forward.fit(X_train, y_train)
forward_features = X.columns[sfs_forward.get_support()]

# Train model on selected features
X_train_fwd = X_train[:, sfs_forward.get_support()]
X_test_fwd = X_test[:, sfs_forward.get_support()]
log_reg.fit(X_train_fwd, y_train)
y_pred_fwd = log_reg.predict(X_test_fwd)
y_prob_fwd = log_reg.predict_proba(X_test_fwd)

forward_acc = accuracy_score(y_test, y_pred_fwd)
forward_auc = roc_auc_score(y_test, y_prob_fwd, multi_class='ovr', average='macro')

# Backward Feature Selection

sfs_backward = SequentialFeatureSelector(
    log_reg,
    direction='backward',
    n_features_to_select='auto',
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_backward.fit(X_train, y_train)
backward_features = X.columns[sfs_backward.get_support()]

# Train model on selected features
X_train_bwd = X_train[:, sfs_backward.get_support()]
X_test_bwd = X_test[:, sfs_backward.get_support()]
log_reg.fit(X_train_bwd, y_train)
y_pred_bwd = log_reg.predict(X_test_bwd)
y_prob_bwd = log_reg.predict_proba(X_test_bwd)

backward_acc = accuracy_score(y_test, y_pred_bwd)
backward_auc = roc_auc_score(y_test, y_prob_bwd, multi_class='ovr', average='macro')


# Results Summary

print("🔹 Forward Selection Features:")
print(list(forward_features))
print(f"Accuracy: {forward_acc:.3f} | AUC: {forward_auc:.3f}\n")

print("🔹 Backward Selection Features:")
print(list(backward_features))
print(f"Accuracy: {backward_acc:.3f} | AUC: {backward_auc:.3f}\n")

# Quick comparison table
results_df = pd.DataFrame({
    'Method': ['Forward', 'Backward'],
    'Selected_Features': [len(forward_features), len(backward_features)],
    'Accuracy': [forward_acc, backward_acc],
    'AUC': [forward_auc, backward_auc]
})

print("🏁 Feature Selection Comparison:")
print(results_df)


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic

🔹 Forward Selection Features:
['CholCheck', 'Stroke', 'HeartDiseaseorAttack', 'Fruits', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'PhysHlth', 'DiffWalk', 'Sex']
Accuracy: 0.727 | AUC: 0.667

🔹 Backward Selection Features:
['HighBP', 'HighChol', 'BMI', 'HeartDiseaseorAttack', 'PhysActivity', 'HvyAlcoholConsump', 'AnyHealthcare', 'GenHlth', 'MentHlth', 'DiffWalk', 'Income']
Accuracy: 0.692 | AUC: 0.768

🏁 Feature Selection Comparison:
     Method  Selected_Features  Accuracy       AUC
0   Forward                 10  0.727472  0.667406
1  Backward                 11  0.692487  0.767915


| Method       | Features Selected | Accuracy |  AUC  |
| :----------- | :---------------: | :------: | :---: |
| **Forward**  |         10        |   0.727  | 0.667 |
| **Backward** |         11        |   0.692  | 0.768 |


Foward feature selection selected 10 features: 'CholCheck', 'Stroke', 'HeartDiseaseorAttack', 'Fruits', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'PhysHlth', 'DiffWalk', 'Sex'. Backward feature selection chose 11 features: 'HighBP', 'HighChol', 'BMI', 'HeartDiseaseorAttack', 'PhysActivity', 'HvyAlcoholConsump', 'AnyHealthcare', 'GenHlth', 'MentHlth', 'DiffWalk', 'Income'.





# Dataset 2 Pima

Forward and Backward Feature selection comparison 

In [9]:
# Forward vs Backward Feature Selection – Logistic Regression

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, roc_auc_score

# --- 1. Prepare data ---
X = df_pima.drop(columns=['Outcome'])
y = df_pima['Outcome']

# --- 2. Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- 4. Define base Logistic Regression model ---
log_reg = LogisticRegression(
    solver='lbfgs', max_iter=5000, multi_class='ovr', random_state=42,  class_weight='balanced',
)

# Forward Feature Selection

sfs_forward = SequentialFeatureSelector(
    log_reg,
    direction='forward',
    n_features_to_select='auto',   # can also set a number like 10
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_forward.fit(X_train, y_train)
forward_features = X.columns[sfs_forward.get_support()]

# Train model on selected features
X_train_fwd = X_train[:, sfs_forward.get_support()]
X_test_fwd = X_test[:, sfs_forward.get_support()]
log_reg.fit(X_train_fwd, y_train)
y_pred_fwd = log_reg.predict(X_test_fwd)
y_prob_fwd = log_reg.predict_proba(X_test_fwd)

forward_acc = accuracy_score(y_test, y_pred_fwd)
forward_auc = roc_auc_score(y_test, y_prob_fwd[:, 1])

# Backward Feature Selection

sfs_backward = SequentialFeatureSelector(
    log_reg,
    direction='backward',
    n_features_to_select='auto',
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs_backward.fit(X_train, y_train)
backward_features = X.columns[sfs_backward.get_support()]

# Train model on selected features
X_train_bwd = X_train[:, sfs_backward.get_support()]
X_test_bwd = X_test[:, sfs_backward.get_support()]
log_reg.fit(X_train_bwd, y_train)
y_pred_bwd = log_reg.predict(X_test_bwd)
y_prob_bwd = log_reg.predict_proba(X_test_bwd)

backward_acc = accuracy_score(y_test, y_pred_bwd)
backward_auc = roc_auc_score(y_test, y_prob_bwd[:, 1])

# Results Summary

print("🔹 Forward Selection Features:")
print(list(forward_features))
print(f"Accuracy: {forward_acc:.3f} | AUC: {forward_auc:.3f}\n")

print("🔹 Backward Selection Features:")
print(list(backward_features))
print(f"Accuracy: {backward_acc:.3f} | AUC: {backward_auc:.3f}\n")

# Quick comparison table
results_df = pd.DataFrame({
    'Method': ['Forward', 'Backward'],
    'Selected_Features': [len(forward_features), len(backward_features)],
    'Accuracy': [forward_acc, backward_acc],
    'AUC': [forward_auc, backward_auc]
})

print("🏁 Feature Selection Comparison:")
print(results_df)


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic

🔹 Forward Selection Features:
['Glucose', 'Insulin', 'BMI', 'DiabetesPedigreeFunction']
Accuracy: 0.701 | AUC: 0.794

🔹 Backward Selection Features:
['Glucose', 'Insulin', 'BMI', 'DiabetesPedigreeFunction']
Accuracy: 0.701 | AUC: 0.794

🏁 Feature Selection Comparison:
     Method  Selected_Features  Accuracy       AUC
0   Forward                  4  0.701299  0.793704
1  Backward                  4  0.701299  0.793704


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic

Both forward and backward feature selection chose the same 4 features: 'Glucose', 'Insulin', 'BMI', and 'DiabetesPedigreeFunction'. They also both achieved the same accuracy of 0.70, and AUC of 0.79. This means that the model is stable and generalizing well and not overfitting. No matter which way the features are selected, the same four variables explain almost all of the predictive signal. These four features have the most clinical significance in the medical setting for predicting diabetes as well. This gives the model more credibility. I wonder why the lasso, ridge and elastic net models did not consider Insulin to be a top feature.  


| Metric       | Use it when...                                       | What it tells you                                       |
| ------------ | ---------------------------------------------------- | ------------------------------------------------------- |
| **Accuracy** | Classes fairly balanced                              | Overall correctness                                     |
| **AUC**      | You want to measure ranking ability (threshold-free) | How well the model separates diabetics vs non-diabetics |
| **F1-score** | Classes imbalanced, and recall matters more          | Balance of precision & recall for diabetics             |
